### Lib

In [3]:
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import pandas as pd
import urllib3
import numpy as np
import pandas as pd
from datetime import date, timedelta
import os
import time
import random

### Czytanie html - download albo local

In [4]:
def load_tge_rdn(
    date_show: str,
    source: str = "request",
    local_filename: str | None = None,
    folder: str = "downloaded_htmls",
) -> tuple[str, BeautifulSoup]:
    """
    Pobiera HTML strony TGE albo odczytuje go z lokalnego pliku.

    Parameters
    ----------
    date_show:
        Data w formacie DD-MM-RRRR, np. "13-07-2026".

    source:
        "request" - pobranie strony z TGE,
        "local"   - odczyt lokalnego pliku HTML.

    local_filename:
        Nazwa lokalnego pliku. Potrzebna przy source="local".
        Plik jest szukany w folderze `downloaded_htmls`.

    folder:
        Folder zawierający lokalne pliki HTML.

    Returns
    -------
    html:
        Surowa zawartość HTML.

    soup:
        Obiekt BeautifulSoup.
    """

    source = source.lower().strip()

    if source == "request":
        urllib3.disable_warnings(
            urllib3.exceptions.InsecureRequestWarning
        )

        session = requests.Session()

        session.headers.update({
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/150.0.0.0 Safari/537.36"
            ),
            "Accept": (
                "text/html,application/xhtml+xml,"
                "application/xml;q=0.9,"
                "image/avif,image/webp,image/apng,*/*;q=0.8"
            ),
            "Accept-Language": "pl-PL,pl;q=0.9,en;q=0.8",
            "Accept-Encoding": "gzip, deflate",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
        })

        # Najpierw wejście na stronę główną, żeby pobrać cookies
        start_response = session.get(
            "https://tge.pl/",
            verify=False,
            timeout=30,
        )

        start_response.raise_for_status()

        url = "https://tge.pl/energia-elektryczna-rdn"

        params = {
            "dateShow": date_show,
            "dateAction": "",
            "type": "2",
        }

        response = session.get(
            url,
            params=params,
            headers={
                "Referer": "https://tge.pl/energia-elektryczna-rdn"
            },
            verify=False,
            timeout=30,
        )

        response.raise_for_status()

        html = response.text

        print(f"Status: {response.status_code}")
        print(f"URL: {response.url}")
        print(f"Długość HTML: {len(html):,} znaków")

    elif source == "local":
        if local_filename is None:
            raise ValueError(
                'Przy source="local" musisz podać local_filename.'
            )

        file_path = Path(folder) / local_filename

        if not file_path.exists():
            raise FileNotFoundError(
                f"Nie znaleziono pliku: {file_path.resolve()}"
            )

        html = file_path.read_text(
            encoding="utf-8",
            errors="replace",
        )

        print(f"Odczytano plik: {file_path.resolve()}")
        print(f"Długość HTML: {len(html):,} znaków")

    else:
        raise ValueError(
            'Argument source musi mieć wartość "request" albo "local".'
        )

    soup = BeautifulSoup(html, "html.parser")

    return html, soup

### Check poprawności html 

In [5]:
import re

def check_df(df):
    # 1. Liczba wierszy
    assert len(df) == 96, f"Oczekiwano 96 wierszy, jest {len(df)}"

    # 2. Liczba kolumn
    expected_columns = 17

    assert df.shape[1] == expected_columns, (
        f"Oczekiwano {expected_columns} kolumn, jest {df.shape[1]}"
    )

    # 3. Dokładne nazwy kolumn
    expected_column_names = [
        "Data dostawy",
        "Typ instrumentu",
        "Fixing I - Kurs [PLN/MWh]",
        "Fixing I - Wolumen [MW]",
        "Notowania ciągłe - Kurs średnioważony [PLN/MWh]",
        "Notowania ciągłe - Wolumen [MW]",
        "Fixing II - Kurs jednolity [EUR/MWh]",
        "Fixing II - Kurs jednolity [PLN/MWh]",
        "Fixing II - Wolumen [MW]",
        "Fixing II - Wolumen kupna [MW]",
        "Fixing II - Wolumen sprzedaży [MW]",
        "Łącznie - Kurs min. [PLN/MWh]",
        "Łącznie - Kurs max. [PLN/MWh]",
        "Łącznie - Kurs średnioważony [PLN/MWh]",
        "Łącznie - Wolumen [MWh]",
        "Łącznie - Wolumen kupna [MWh]",
        "Łącznie - Wolumen sprzedaży [MWh]",
    ]

    assert df.columns.tolist() == expected_column_names, (
        "Nazwy albo kolejność kolumn nie zgadzają się"
    )

    # 4. Brak duplikatów interwałów
    assert not df["Data dostawy"].duplicated().any(), (
        "Występują zduplikowane interwały"
    )

    # 5. Format interwałów
    pattern = r"^\d{4}-\d{2}-\d{2}_Q(?:[01]\d|2[0-3]):(?:00|15|30|45)$|^\d{4}-\d{2}-\d{2}_Q24:00$"

    invalid_periods = df.loc[
        ~df["Data dostawy"].str.match(pattern, na=False),
        "Data dostawy"
    ]

    assert invalid_periods.empty, (
        f"Niepoprawne oznaczenia interwałów:\n{invalid_periods.tolist()}"
    )

    # 6. Wszystkie interwały dotyczą tej samej daty
    dates = df["Data dostawy"].str.extract(r"^(\d{4}-\d{2}-\d{2})")[0]

    assert dates.nunique() == 1, (
        f"W danych występuje więcej niż jedna data: {dates.unique().tolist()}"
    )

    # 7. Typ instrumentu powinien być wszędzie 15
    # lub inna wartość zgodna z tym, co faktycznie zwraca strona
    unexpected_types = df.loc[
        df["Typ instrumentu"].astype(str).str.strip() != "15",
        "Typ instrumentu"
    ]

    assert unexpected_types.empty, (
        f"Nieoczekiwany typ instrumentu: {unexpected_types.unique().tolist()}"
    )

    # 8. Kolumny liczbowe rzeczywiście są liczbowe
    numeric_columns = df.columns[2:]

    non_numeric = [
        column
        for column in numeric_columns
        if not pd.api.types.is_numeric_dtype(df[column])
    ]

    assert not non_numeric, (
        f"Kolumny nie są numeryczne: {non_numeric}"
    )

    # 9. Sprawdzenie, czy nie wszystko zamieniło się na NaN
    all_empty_columns = [
        column
        for column in numeric_columns
        if df[column].isna().all()
    ]

    assert not all_empty_columns, (
        f"Kolumny całkowicie puste: {all_empty_columns}"
    )

    # 10. Nietypowo duża liczba braków
    missing_share = df[numeric_columns].isna().mean()

    too_many_missing = missing_share[missing_share > 0.95]

    assert too_many_missing.empty, (
        f"Ponad 95% braków w kolumnach:\n{too_many_missing}"
    )

    print("Wszystkie checki przeszły poprawnie.")

    return df

In [6]:
def generate_df_from_soup(soup, date):

    #format daty "2026-07-14"
    print("czytany plik z dnia: ", date)
    rows = []

    start = datetime.strptime(f"{date} 00:15", "%Y-%m-%d %H:%M")

    for i in range(96):
        current = start + timedelta(minutes=15 * i)

        if i == 95:
            period = f"{date}_Q24:00"
        else:
            period = f"{date}_Q{current.hour:02d}:{current.minute:02d}"

        master_tag = soup.find(
            "td",
            string=lambda text, period=period:
                text and text.strip() == period
        )

        if master_tag is None:
            print(f"Nie znaleziono: {period}")
            continue

        tr = master_tag.find_parent("tr")

        values = [
            td.get_text(" ", strip=True)
            for td in tr.find_all("td")
        ]

        rows.append(values)

    df = pd.DataFrame(rows)

    columns = [
    "Data dostawy",
    "Typ instrumentu",

    # Fixing I
    "Fixing I - Kurs [PLN/MWh]",
    "Fixing I - Wolumen [MW]",

    # Notowania ciągłe
    "Notowania ciągłe - Kurs średnioważony [PLN/MWh]",
    "Notowania ciągłe - Wolumen [MW]",

    # Fixing II
    "Fixing II - Kurs jednolity [EUR/MWh]",
    "Fixing II - Kurs jednolity [PLN/MWh]",
    "Fixing II - Wolumen [MW]",
    "Fixing II - Wolumen kupna [MW]",
    "Fixing II - Wolumen sprzedaży [MW]",

    # Łącznie notowania
    "Łącznie - Kurs min. [PLN/MWh]",
    "Łącznie - Kurs max. [PLN/MWh]",
    "Łącznie - Kurs średnioważony [PLN/MWh]",
    "Łącznie - Wolumen [MWh]",
    "Łącznie - Wolumen kupna [MWh]",
    "Łącznie - Wolumen sprzedaży [MWh]",
    ]

    df = pd.DataFrame(rows, columns=columns)

    columns_to_convert = df.columns[2:]

    df[columns_to_convert] = (
        df[columns_to_convert]
        .replace(r"^\s*-\s*$", "0", regex=True)
        .apply(
            lambda col: pd.to_numeric(
                col.astype("string")
                .str.replace("\xa0", "", regex=False)
                .str.replace(" ", "", regex=False)
                .str.replace(",", ".", regex=False),
                errors="coerce"
            )
        )
    )

    check_df(df)

    return df

In [7]:
def generate_hourly_df(df, date_show):
    fixing_1_price = "Fixing I - Kurs [PLN/MWh]"
    fixing_1_volume = "Fixing I - Wolumen [MW]"

    fixing_2_price = "Fixing II - Kurs jednolity [PLN/MWh]"
    fixing_2_volume = "Fixing II - Wolumen [MW]"

    df = df.reset_index(drop=True)
    df["grupa_godzinowa"] = np.arange(len(df)) // 4

    hourly = (
        df.groupby("grupa_godzinowa")
        .apply(
            lambda group: pd.Series({
                "Fixing I - Kurs średnioważony [PLN/MWh]": (
                    (group[fixing_1_price] * group[fixing_1_volume]).sum()
                    / group[fixing_1_volume].sum()
                    if group[fixing_1_volume].sum() != 0
                    else np.nan
                ),

                "Fixing I - Wolumen [MWh]":
                    group[fixing_1_volume].sum() / 4,

                "Fixing II - Kurs średnioważony [PLN/MWh]": (
                    (group[fixing_2_price] * group[fixing_2_volume]).sum()
                    / group[fixing_2_volume].sum()
                    if group[fixing_2_volume].sum() != 0
                    else np.nan
                ),

                "Fixing II - Wolumen [MWh]":
                    group[fixing_2_volume].sum() / 4,
            }),
            include_groups=False
        )
        .reset_index(drop=True)
    )

    hourly.insert(
        0,
        "gg",
        [f"{hour:02d}" for hour in range(1, len(hourly) + 1)]
)
    hourly.insert(0, "rr-mm-dd", date_show)
    
    return hourly

### Loop over dates

In [8]:
def reteirve_df(html_date, d_source):
    
    if isinstance(html_date, str) and len(html_date) == 8 and html_date.isdigit():
        date_show = f"{html_date[:4]}-{html_date[4:6]}-{html_date[6:8]}"
        date_trading = (
            datetime.strptime(date_show, "%Y-%m-%d") - timedelta(days=1)
        ).strftime("%d-%m-%Y")
    else:
        raise ValueError("Data htmla musi być stringiem w formacie YYYYMMDD")


    html, soup = load_tge_rdn(
    date_show=date_trading,
    source=d_source, #local lub request
    local_filename=f"TGE - Rynek Dnia Następnego_{html_date}.html"
    )

    df = generate_df_from_soup(soup, date_show)

    df_hourly = generate_hourly_df(df, date_show)

    return df, df_hourly


In [ ]:
s_date_input = os.environ["S_DATE_INPUT"]
e_date_input = os.environ["E_DATE_INPUT"]
source_type_input = os.environ["SOURCE_TYPE_INPUT"]

s_year, s_month, s_day = map(int, s_date_input.split("."))
e_year, e_month, e_day = map(int, e_date_input.split("."))

match source_type_input:
    case "1": source_type = "local"
    case "2": source_type = "request"

start_date = date(s_year, s_month, s_day)
end_date = date(e_year, e_month, e_day)

period_str = str(start_date) + "_" + str(end_date)

results = []
results_hourly = []

current_date = start_date

while current_date <= end_date:
    date_str = current_date.strftime("%Y%m%d")

    print(f"Przetwarzam: {date_str}")

    try:
        df_day, df_day_hourly = reteirve_df(date_str, source_type)
        results.append(df_day)
        results_hourly.append(df_day_hourly)

    except Exception as error:
        print(f"Błąd dla {date_str}: {error}")

    current_date += timedelta(days=1)

    delay = random.uniform(4, 10)
    print(f"Czekam {delay:.1f} s...")
    time.sleep(delay)

df_all = pd.concat(results, ignore_index=True)
df_hourly_all = pd.concat(results_hourly, ignore_index=True)


#czas dostawy
time_part = df_all["Data dostawy"].str.extract(r"_Q(\d{2}):(\d{2})")

hour = time_part[0].astype(int)
minute = time_part[1].astype(int)

df_all["Początek dostawy w godzinie"] = hour

df_all["Koniec dostawy w godzinie"] = hour + 1

# Dla pełnych godzin, np. Q01:00, interwał nadal należy do godziny 00–01
mask_full_hour = minute.eq(0)

df_all.loc[mask_full_hour, "Początek dostawy w godzinie"] = (
    hour[mask_full_hour] - 1
)

df_all.loc[mask_full_hour, "Koniec dostawy w godzinie"] = hour[mask_full_hour]


### do excela
with pd.ExcelWriter(f"RDN_{period_str}.xlsx", engine="openpyxl") as writer:
    df_all.to_excel(writer, sheet_name="RDN_15min", index=False)
    df_hourly_all.to_excel(writer, sheet_name="RDN_60min_średnie_ważone", index=False)

Przetwarzam: 20260801


Status: 200
URL: https://tge.pl/energia-elektryczna-rdn?dateShow=31-07-2026&dateAction=&type=2
Długość HTML: 361,416 znaków
czytany plik z dnia:  2026-08-01


Wszystkie checki przeszły poprawnie.
Czekam 5.1 s...


Przetwarzam: 20260802


Status: 200
URL: https://tge.pl/energia-elektryczna-rdn?dateShow=01-08-2026&dateAction=&type=2
Długość HTML: 360,951 znaków
czytany plik z dnia:  2026-08-02
Wszystkie checki przeszły poprawnie.
Czekam 8.9 s...


...

Status: 200
URL: https://tge.pl/energia-elektryczna-rdn?dateShow=30-08-2026&dateAction=&type=2
Długość HTML: 361,668 znaków
czytany plik z dnia:  2026-08-31


Wszystkie checki przeszły poprawnie.
Czekam 5.6 s...


Przetwarzam: 20260901


Status: 200
URL: https://tge.pl/energia-elektryczna-rdn?dateShow=31-08-2026&dateAction=&type=2
Długość HTML: 361,349 znaków
czytany plik z dnia:  2026-09-01


Wszystkie checki przeszły poprawnie.
Czekam 9.8 s...
